In [2]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from scipy.signal import savgol_filter
from scipy.spatial import distance
from librosa.sequence import dtw
from tqdm import trange, tqdm
import networkx as nx
from descartes import PolygonPatch
from shapely.geometry import Polygon, MultiPolygon, Point, MultiLineString, LineString, shape, JOIN_STYLE
from shapely.geometry.polygon import LinearRing
from shapely.ops import snap, unary_union
import h5py

In [67]:
cd /Users/zoltan/Dropbox/Channels/meanderpy/meanderpy

/Users/zoltan/Dropbox/Channels/meanderpy/meanderpy


In [68]:
import meanderpy as mp

In [44]:
cd /Users/zoltan/Dropbox/meandergraph

/Users/zoltan/Dropbox/Meandergraph


In [45]:
import meandergraph as mg

In [69]:
nit = 2000                   # number of iterations
W = 200.0                    # channel width (m)
D = 6.0                      # channel depth (m)
depths = D * np.ones((nit,))  # channel depths for different iterations  
pad = 100                    # padding (number of nodepoints along centerline)
deltas = 50.0                # sampling distance along centerline           
Cfs = 0.011 * np.ones((nit,)) # dimensionless Chezy friction factor
crdist = 2 * W               # threshold distance at which cutoffs occur
kl = 60.0/(365*24*60*60.0)   # migration rate constant (m/s)
kv =  1.0e-12               # vertical slope-dependent erosion rate constant (m/s)
dt = 2*0.05*365*24*60*60.0     # time step (s)
dens = 1000                  # density of water (kg/m3)
saved_ts = 10                # which time steps will be saved
n_bends = 30                 # approximate number of bends you want to model
Sl = 0.0                     # initial slope (matters more for submarine channels than rivers)
t1 = 500                    # time step when incision starts
t2 = 700                    # time step when lateral migration starts
t3 = 1200                    # time step when aggradation starts
aggr_factor = 2e-9         # aggradation factor (m/s, about 0.18 m/year, it kicks in after t3)

# ch = mp.generate_initial_channel(W, depths[0], Sl, deltas, pad, n_bends) # initialize channel

fname = '/Users/zoltan/Dropbox/Meandergraph/meandergraph_example_model_1_starting_xy.npz'
data = np.load(fname)
x = data['arr_0']
y = data['arr_1']
z = np.zeros(len(x))

ch = mp.Channel(x, y, z, W, D)
chb = mp.ChannelBelt(channels=[ch], cutoffs=[], cl_times=[0.0], cutoff_times=[]) # create channel belt object

chb.migrate(nit,saved_ts,deltas,pad,crdist,depths,Cfs,kl,kv,dt,dens,t1,t2,t3,aggr_factor) # channel migration
fig = chb.plot('strat', 20, 60, chb.cl_times[-1], len(chb.channels)) # plotting

100%|██████████| 2000/2000 [00:49<00:00, 40.42it/s]


In [70]:
points = [(15271.731586685313, -67.20062160695579),
 (28052.115239837553, 187.62297883933115)]

In [71]:
from scipy import signal, spatial
cl_points = np.vstack((chb.channels[0].x, chb.channels[0].y)).T # coordinates of first centerline
tree = spatial.KDTree(cl_points)
plt.plot(chb.channels[0].x[tree.query(points[0])[1]], chb.channels[0].y[tree.query(points[0])[1]], 
         'ro', zorder=10000)
plt.plot(chb.channels[0].x[tree.query(points[1])[1]], chb.channels[0].y[tree.query(points[1])[1]], 
         'ro', zorder=10000)

In [ ]:
first_index = tree.query(points[0])[1]
last_index = tree.query(points[1])[1]

first_channel = 60
last_channel = 200

# create lists of x, y, z coordinates from channel belt object:
X = []
Y = []
Z = []
for i in range(first_channel, last_channel):
    X.append(chb.channels[i].x)
    Y.append(chb.channels[i].y)
    Z.append(chb.channels[i].z)

# correlate all centerlines:    
P = []
Q = []
for i in trange(len(X) - 1):
    p, q, _ = mg.correlate_curves(X[i], X[i+1], Y[i], Y[i+1])
    P.append(p)
    Q.append(q)

indices1, x, y = mg.find_indices(first_index, X, Y, P, Q)
indices2, x, y = mg.find_indices(last_index, X, Y, P, Q)
for i in range(len(X)):
    X[i] = X[i][indices1[i] : indices2[i]+1]
    Y[i] = Y[i][indices1[i] : indices2[i]+1]
    Z[i] = Z[i][indices1[i] : indices2[i]+1]

# create bank coordinates:
X1 = [] # right bank x coordinate
X2 = [] # left bank x coordinate
Y1 = [] # right bank y coordinate
Y2 = [] # left bank y coordinate
for i in range(len(X)):
    x1 = X[i].copy()
    y1 = Y[i].copy()
    x2 = X[i].copy()
    y2 = Y[i].copy()
    x = X[i].copy()
    y = Y[i].copy()
    ns = len(x)
    dx = np.diff(x); dy = np.diff(y) 
    ds = np.sqrt(dx**2+dy**2)
    x1[:-1] = x[:-1] + 0.5*W*np.diff(y)/ds
    y1[:-1] = y[:-1] - 0.5*W*np.diff(x)/ds
    x2[:-1] = x[:-1] - 0.5*W*np.diff(y)/ds
    y2[:-1] = y[:-1] + 0.5*W*np.diff(x)/ds
    x1[ns-1] = x[ns-1] + 0.5*W*(y[ns-1]-y[ns-2])/ds[ns-2]
    y1[ns-1] = y[ns-1] - 0.5*W*(x[ns-1]-x[ns-2])/ds[ns-2]
    x2[ns-1] = x[ns-1] - 0.5*W*(y[ns-1]-y[ns-2])/ds[ns-2]
    y2[ns-1] = y[ns-1] + 0.5*W*(x[ns-1]-x[ns-2])/ds[ns-2]
    X1.append(x1)
    X2.append(x2)
    Y1.append(y1)
    Y2.append(y2)
    
# resample centerlines to ds = 1.0 meters:
for i in range(len(X)):
    x,y,z,dx,dy,dz,ds,s = mp.resample_centerline(X[i], Y[i], Z[i], 1.0)
    X[i] = x
    Y[i] = y
for i in range(len(X1)):
    x,y,z,dx,dy,dz,ds,s = mp.resample_centerline(X1[i], Y1[i], Z[i], 1.0)
    X1[i] = x
    Y1[i] = y
for i in range(len(X2)):
    x,y,z,dx,dy,dz,ds,s = mp.resample_centerline(X2[i], Y2[i], Z[i], 1.0)
    X2[i] = x
    Y2[i] = y

P, Q = mg.correlate_set_of_curves(X, Y)
P1, Q1 = mg.correlate_set_of_curves(X1, Y1)
P2, Q2 = mg.correlate_set_of_curves(X2, Y2)

100%|██████████| 139/139 [2:00:20<00:00, 51.95s/it] 


In [111]:
ts = len(X)
graph = mg.create_graph_from_channel_lines(X[:ts], Y[:ts], P[:ts-1], Q[:ts-1], 
                                           n_points=40, max_dist=100, remove_cutoff_edges=True)
graph1 = mg.create_graph_from_channel_lines(X1[:ts], Y1[:ts], P1[:ts-1], Q1[:ts-1], 
                                            n_points=40, max_dist=100, remove_cutoff_edges=True)
graph2 = mg.create_graph_from_channel_lines(X2[:ts], Y2[:ts], P2[:ts-1], Q2[:ts-1], 
                                            n_points=40, max_dist=100, remove_cutoff_edges=True)

100%|██████████| 140/140 [00:00<00:00, 294.14it/s]


In [99]:
graph = mg.remove_high_density_nodes(graph, min_dist = 20, max_dist = 60)
graph1 = mg.remove_high_density_nodes(graph1, min_dist = 20, max_dist = 60)
graph2 = mg.remove_high_density_nodes(graph2, min_dist = 20, max_dist = 60)

100%|██████████| 140/140 [00:10<00:00, 12.75it/s]


In [ ]:
cutoff_area = 0.25*1e6
fig = plt.figure()
ax = fig.add_subplot(111)
scrolls, scroll_ages, cutoffs, all_bars_graph = mg.create_scrolls_and_find_connected_scrolls(graph1, graph2, cutoff_area)
plt.axis('equal');

In [ ]:
# create bars and bar graphs
min_area = 5000
wbars, poly_graph_1, poly_graph_2 = mg.create_polygon_graphs_and_bar_graphs(graph1, graph2, all_bars_graph, 
                                            scrolls, scroll_ages, X1, Y1, X2, Y2, min_area)

100%|██████████| 25/25 [00:45<00:00,  1.82s/it]


In [102]:
# plot bar polygons and their numbers
fig = plt.figure(figsize = (15, 9))
ax = fig.add_subplot(111)
for wbar in wbars:
    ax.add_patch(PolygonPatch(wbar.polygon))
count = 0
for wbar in wbars:
    ax.text(wbar.polygon.centroid.x, wbar.polygon.centroid.y, str(count), fontsize = 16)
    count += 1
plt.axis('equal');

/Users/zoltan/anaconda3/envs/meanderpy/lib/python3.7/site-packages/descartes/patch.py:64: ShapelyDeprecationWarning: The array interface is deprecated and will no longer work in Shapely 2.0. Convert the '.coords' to a numpy array instead.
  [asarray(r)[:, :2] for r in t.interiors])


In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
for i in range(len(wbars)):
    if i not in [0, 1, 2, 6]:
        mg.plot_migration_rate_map(wbars[i], graph1, graph2, vmin=0, vmax=30, ax=ax)
plt.axis('equal');

In [ ]:
fig = plt.figure(figsize = (20, 12))
ax = fig.add_subplot(111)
mg.plot_bar_graphs(graph1, graph2, wbars, cutoffs, dt, X1[:ts], Y1[:ts], X2[:ts], Y2[:ts], W, 0, 40, 'migration', ax)
ax.set_adjustable("box")
ax.axis('equal')
fig.tight_layout()

100%|██████████| 25/25 [00:27<00:00,  1.09s/it]


In [109]:
ax.plot(X1[ts-1], Y1[ts-1], 'r') # this is needed because Adobe Illustrator does not like the most recent channel polygon
ax.plot(X2[ts-1], Y2[ts-1], 'r');

In [96]:
fig = plt.figure()
ax = fig.add_subplot(111)
mg.plot_graph(graph, ax)
plt.axis('equal');

In [112]:
cd /Users/zoltan/Dropbox/Meandergraph

/Users/zoltan/Dropbox/Meandergraph


### Write all data to HDF5 file

In [123]:
def write_hdf5_dataset(list_of_arrays, list_name, f):
    grp = f.create_group(list_name)
    for i in range(len(list_of_arrays)):
        grp.create_dataset(str(i), data = list_of_arrays[i])

lists_of_arrays = [P, Q, P1, Q1, P2, Q2, X, Y, X1, Y1, X2, Y2]
list_names = ['P', 'Q', 'P1', 'Q1', 'P2', 'Q2', 'X', 'Y', 'X1', 'Y1', 'X2', 'Y2']

fname = 'meandergraph_example_model_1_all_data.hdf5'
f = h5py.File(fname,'w')
for list_of_arrays, list_name in zip(lists_of_arrays, list_names):
    write_hdf5_dataset(list_of_arrays, list_name, f)
f.close()

In [55]:
%matplotlib qt

In [85]:
from importlib import reload
reload(mg)

<module 'meandergraph' from '/Users/zoltan/Dropbox/Meandergraph/meandergraph/meandergraph.py'>